In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchaudio
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd
import random
from Levenshtein import distance as levenshtein_distance  # pip install python-Levenshtein

# -----------------------------------------
# 1. Алфавит и кодирование (без изменений)
# -----------------------------------------
alphabet = {
    '<pad>': 0, ' ': 1, '#': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20,
    'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28,
    'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36,
    'Ш': 37, 'Щ': 38, 'Ъ': 39, 'Ы': 40, 'Ь': 41, 'Э': 42, 'Ю': 43, 'Я': 44
}
num_classes = len(alphabet)

def encode_label(text: str, alpha: dict) -> list:
    return [alpha[ch] for ch in text if ch in alpha]

# --------------------------------------------------
# 2. AudioTransform: torchaudio-based feature extractor
# --------------------------------------------------
class AudioTransform(nn.Module):
    def __init__(self, sr=16000, n_mels=128, n_fft=1024, hop_length=512):
        super().__init__()
        self.mel_spec = MelSpectrogram(sample_rate=sr, n_fft=n_fft,
                                       hop_length=hop_length, n_mels=n_mels)
        self.to_db = AmplitudeToDB()

    def forward(self, waveform: torch.Tensor):
        mel = self.mel_spec(waveform)
        mel_db = self.to_db(mel)
        return mel_db

# --------------------------------------------------
# 3. Dataset with global normalization & augmentation
# --------------------------------------------------
class MorseAudioCTCDataset(Dataset):
    def __init__(self, df, transform, alpha, augment=False, mean=0.0, std=1.0):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.alpha = alpha
        self.augment = augment
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        path = f"morse_dataset/{row['id']}"
        waveform, sr = torchaudio.load(path)

        # Augmentation on waveform
        if self.augment and random.random() < 0.5:
            noise = torch.randn_like(waveform) * 0.002
            waveform = waveform + noise

        # Feature extraction
        mel = self.transform(waveform)

        # Global normalization
        mel = (mel - self.mean) / (self.std + 1e-6)

        # Encode label
        label = torch.tensor(encode_label(row['message'], self.alpha), dtype=torch.long)
        return mel, label, mel.shape[-1]

# --------------------------------------------------
# 4. Collate fn for CTC
# --------------------------------------------------
def ctc_collate_fn(batch, pool_factor=4):
    mels, labels, lengths = zip(*batch)
    max_T = max([m.size(-1) for m in mels])
    padded = []
    for m in mels:
        pad = max_T - m.size(-1)
        padded.append(F.pad(m, (0, pad)))
    audio_batch = torch.stack(padded)

    labels_concat = torch.cat(labels)
    input_lengths = torch.tensor([l // pool_factor for l in lengths], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

# --------------------------------------------------
# 5. Model (CNN + BiLSTM) — скорректированный пулинг по частоте
# --------------------------------------------------
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes=45, in_channels=1,
                 n_mels=128, lstm_hidden=384, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )
        self.dropout = nn.Dropout(dropout)
        self.rnn_input_dim = (n_mels // 4) * 128
        self.lstm = nn.LSTM(
            self.rnn_input_dim, lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True, dropout=dropout
        )
        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * f)
        x = self.dropout(x)
        x, _ = self.lstm(x)
        x = self.classifier(x)
        return x.permute(1, 0, 2)

# --------------------------------------------------
# 6. Greedy decoder + Levenshtein eval
# --------------------------------------------------
def greedy_decode(logits):
    pred = logits.argmax(-1).transpose(0, 1)
    sentences = []
    for seq in pred:
        prev = None
        chars = []
        for idx in seq.cpu().numpy():
            if idx != prev and idx != 0:
                chars.append(list(alphabet.keys())[list(alphabet.values()).index(idx)])
            prev = idx
        sentences.append("".join(chars))
    return sentences

# --------------------------------------------------
# 7. Training loop with AdamW, scheduler, clipping, CER
# --------------------------------------------------
def train_ctc(model, train_loader, val_loader,
              epochs=20, lr=1e-3, weight_decay=1e-5,
              device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)
    ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for aud, labels, in_lens, tgt_lens in tqdm(train_loader, desc=f"Train ep{ep}"):
            aud, labels = aud.to(device), labels.to(device)
            in_lens, tgt_lens = in_lens.to(device), tgt_lens.to(device)

            optimizer.zero_grad()
            logits = model(aud)
            loss = ctc_loss(F.log_softmax(logits, 2), labels, in_lens, tgt_lens)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        total_dist = 0
        count = 0
        with torch.no_grad():
            for aud, labels, in_lens, tgt_lens in tqdm(val_loader, desc="Val"):
                aud, labels = aud.to(device), labels.to(device)
                in_lens, tgt_lens = in_lens.to(device), tgt_lens.to(device)
                logits = model(aud)
                val_loss += ctc_loss(F.log_softmax(logits, 2), labels, in_lens, tgt_lens).item()

                preds = greedy_decode(logits)
                truths = []
                idx_flat = 0
                for length in tgt_lens.cpu().tolist():
                    seq = labels[idx_flat:idx_flat + length].cpu().tolist()
                    truths.append(
                        "".join(list(alphabet.keys())[list(alphabet.values()).index(i)] for i in seq)
                    )
                    idx_flat += length
                for p, t in zip(preds, truths):
                    total_dist += levenshtein_distance(p, t)
                    count += 1

        avg_tr = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        cer = total_dist / count
        scheduler.step(avg_val)
        print(f"Ep{ep}: TrainL={avg_tr:.4f}, ValL={avg_val:.4f}, CER={cer:.4f}")

# --------------------------------------------------
# 8. Main
# --------------------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    feat_ext = AudioTransform(sr=16000)

    # Вычисляем глобальные mean/std
    stats_ds = MorseAudioCTCDataset(train_df, feat_ext, alphabet, augment=False)
    all_mels = []
    for m, _, _ in tqdm(stats_ds, desc="Compute stats"):
        all_mels.append(m)
    stacked = torch.cat([m.flatten() for m in all_mels])
    global_mean = stacked.mean()
    global_std = stacked.std()
    print(f"Global mean={global_mean:.4f}, std={global_std:.4f}")

    train_ds = MorseAudioCTCDataset(train_df, feat_ext, alphabet,
                                    augment=True, mean=global_mean, std=global_std)
    val_ds = MorseAudioCTCDataset(val_df, feat_ext, alphabet,
                                  augment=False, mean=global_mean, std=global_std)

    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_factor=4)
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=8,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_factor=4)
    )

    model = CNNBiLSTMCTC(
        num_classes=num_classes,
        lstm_hidden=256,
        lstm_layers=3,
        dropout=0.3
    )

    train_ctc(
        model,
        train_loader,
        val_loader,
        epochs=33,
        lr=1e-3
    )


Compute stats: 100%|██████████| 24000/24000 [06:30<00:00, 61.52it/s]


KeyError: 24000

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchaudio
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd
import random
from Levenshtein import distance as levenshtein_distance  # pip install python-Levenshtein

# -----------------------------------------
# 1. Алфавит и кодирование
# -----------------------------------------
alphabet = {
    '<pad>': 0, ' ': 1, '#': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20,
    'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28,
    'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36,
    'Ш': 37, 'Щ': 38, 'Ъ': 39, 'Ы': 40, 'Ь': 41, 'Э': 42, 'Ю': 43, 'Я': 44
}
num_classes = len(alphabet)

def encode_label(text: str, alpha: dict) -> list:
    return [alpha[ch] for ch in text if ch in alpha]

# -----------------------------------------
# 2. AudioTransform: torchaudio-based
# -----------------------------------------
class AudioTransform(nn.Module):
    def __init__(self, sr=16000, n_mels=128, n_fft=1024, hop_length=512):
        super().__init__()
        self.mel_spec = MelSpectrogram(sample_rate=sr, n_fft=n_fft,
                                       hop_length=hop_length, n_mels=n_mels)
        self.to_db = AmplitudeToDB()

    def forward(self, waveform: torch.Tensor):
        mel = self.mel_spec(waveform)
        mel_db = self.to_db(mel)
        return mel_db

# --------------------------------------------------
# 3. Precomputed Dataset (all audio same length)
# --------------------------------------------------
class PrecomputedMorseDataset(Dataset):
    def __init__(self, df, transform, alpha, mean, std, augment=False):
        self.alpha = alpha
        self.augment = augment
        self.mean = mean
        self.std = std
        waves = []
        labels = []
        for _, row in df.iterrows():
            wav, sr = torchaudio.load(f"morse_dataset/{row['id']}")  # [1, L]
            # augmentation
            if augment and random.random() < 0.5:
                wav = wav + torch.randn_like(wav) * 0.002
            mel = transform(wav)  # [1, n_mels, T]
            mel = (mel - mean) / (std + 1e-6)
            waves.append(mel)
            lbl = torch.tensor(encode_label(row['message'], alpha), dtype=torch.long)
            labels.append(lbl)
        self.mels = torch.stack(waves)  # [N,1,n_mels,T]
        self.labels = labels
        # concat targets and lens
        self.concat = torch.cat(labels)
        self.tgt_lens = torch.tensor([l.size(0) for l in labels], dtype=torch.long)
        # fixed input length after pooling /4
        self.in_len = self.mels.size(-1) // 4

    def __len__(self):
        return len(self.mels)

    def __getitem__(self, idx):
        return self.mels[idx], self.labels[idx], self.in_len, self.tgt_lens[idx]

# --------------------------------------------------
# 4. Model: CNN + BiLSTM
# --------------------------------------------------
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes=45, in_channels=1,
                 n_mels=128, lstm_hidden=384, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128,3,1,1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2,1)),
        )
        self.dropout = nn.Dropout(dropout)
        self.rnn_input = (n_mels//4)*128
        self.lstm = nn.LSTM(self.rnn_input, lstm_hidden, num_layers=lstm_layers,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.classifier = nn.Linear(lstm_hidden*2, num_classes)

    def forward(self, x):  # x: [B,1,n_mels,T]
        y = self.cnn(x)  # [B,128,n_mels/4,T/4]
        b,c,f,t = y.size()
        y = y.permute(0,3,1,2).reshape(b, t, c*f)
        y = self.dropout(y)
        y, _ = self.lstm(y)
        y = self.classifier(y)
        return y.permute(1,0,2)  # [T,B,C]

# --------------------------------------------------
# 5. Greedy decoder & CER
# --------------------------------------------------
def greedy_decode(logits):
    seqs = logits.argmax(-1).transpose(0,1)
    out = []
    chars = list(alphabet.keys())
    for s in seqs:
        prev = None; string = []
        for idx in s.cpu().tolist():
            if idx!=prev and idx!=0:
                string.append(chars[idx])
            prev = idx
        out.append("".join(string))
    return out

# --------------------------------------------------
# 6. Training loop
# --------------------------------------------------
def train_ctc(model, tr_ld, va_ld, epochs=20, lr=1e-3, wd=1e-5, device=None):
    print(">>> старт training loop")
    dev = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(dev)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt,'min',patience=2)
    loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

    for ep in range(1, epochs+1):
        model.train(); tot=0
        for x, lbl, il, tl in tqdm(tr_ld, desc=f"Train{ep}"):
            x,lbl = x.to(dev), lbl.to(dev)
            il,tl = il.to(dev), tl.to(dev)
            opt.zero_grad()
            logits = model(x)
            loss = loss_fn(F.log_softmax(logits,2), lbl, il, tl)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); tot+=loss.item()
        # validation
        model.eval(); valL=0; dist=0; cnt=0
        with torch.no_grad():
            for x,lbl,il,tl in tqdm(va_ld, desc="Val"):
                x,lbl = x.to(dev), lbl.to(dev)
                il,tl = il.to(dev), tl.to(dev)
                logits = model(x)
                valL+=loss_fn(F.log_softmax(logits,2), lbl, il, tl).item()
                preds = greedy_decode(logits)
                # reconstruct labels
                ptr=0
                for tlen in tl.cpu().tolist():
                    truth = "".join(list(alphabet.keys())[i] for i in lbl[ptr:ptr+tlen].cpu().tolist())
                    ptr+=tlen
                    dist+=levenshtein_distance(preds[cnt], truth); cnt+=1
        print(f"Ep{ep}: TrL={tot/len(tr_ld):.4f}, VaL={valL/len(va_ld):.4f}, CER={dist/cnt:.4f}")
        sched.step(valL/len(va_ld))

# --------------------------------------------------
# 7. Main
# --------------------------------------------------
if __name__=="__main__":
    df = pd.read_csv("train.csv")
    tr_df, va_df = train_test_split(df, test_size=0.2, random_state=42)

    feat = AudioTransform(sr=16000)
    # stats
    temp = [feat(torchaudio.load(f"morse_dataset/{i}")[0]) for i in tr_df['id']]
    allm = torch.cat([t.flatten() for t in temp])
    mean, std = allm.mean(), allm.std()

    tr_ds = PrecomputedMorseDataset(tr_df, feat, alphabet, mean, std, augment=True)
    va_ds = PrecomputedMorseDataset(va_df, feat, alphabet, mean, std, augment=False)

    tr_ld = DataLoader(tr_ds, batch_size=16, shuffle=True, num_workers=0)
    va_ld = DataLoader(va_ds, batch_size=8, shuffle=False, num_workers=0)

    model = CNNBiLSTMCTC(num_classes=num_classes, lstm_hidden=256, lstm_layers=3, dropout=0.3)
    train_ctc(model, tr_ld, va_ld, epochs=33, lr=1e-3)


KeyboardInterrupt: 